# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd

# Cargar los 3 archivos
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# Estandarizar nombres de columnas (minúsculas, sin espacios)
for df in [df1, df2, df3]:
    df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Combinar los 3 dataframes
df_combined = pd.concat([df1, df2, df3], ignore_index=True)
print(f"Shape combinado: {df_combined.shape}")

# Limpieza 
# Gender
gender_map = {"male": "M", "m": "M", "femal": "F", "female": "F", "f": "F"}
df_combined["gender"] = df_combined["gender"].str.strip().str.lower().map(gender_map)

# State
df_combined["st"] = df_combined["st"].replace({"AZ": "Arizona", "WA": "Washington", "Cali": "California"})

# Education
df_combined["education"] = df_combined["education"].replace({"Bachelors": "Bachelor"})

# CLV: quitar % y convertir a float
df_combined["customer_lifetime_value"] = (
    df_combined["customer_lifetime_value"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Vehicle class
df_combined["vehicle_class"] = df_combined["vehicle_class"].replace({
    "Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"
})

# Nulos: numéricos → media, categóricos → moda
for col in df_combined.select_dtypes(include="number").columns:
    df_combined[col] = df_combined[col].fillna(df_combined[col].mean())

for col in df_combined.select_dtypes(include="object").columns:
    df_combined[col] = df_combined[col].fillna(df_combined[col].mode()[0])

# Duplicados
df_combined = df_combined.drop_duplicates().reset_index(drop=True)


Shape combinado: (12074, 12)


In [2]:
print(f"Shape final limpio: {df_combined.shape}")
df_combined.head()

Shape final limpio: (9135, 12)


,customer,st,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount,state
0,RB50392,Washington,F,Master,7.799463e+05,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934,California
1,QZ44356,Arizona,F,Bachelor,6.979536e+05,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935,California
2,AI49188,Nevada,F,Bachelor,1.288743e+06,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247,California
3,WW63253,California,M,Bachelor,7.645862e+05,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344,California
4,GA49547,Washington,M,High School or Below,5.363077e+05,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323,California


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

# Siempre verificar primero
print(df.shape)
print(df.columns.tolist())
print(df.head())

(10910, 27)
['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']
   unnamed:_0 customer       state  customer_lifetime_value response  \
0           0  DK49336     Arizona              4809.216960       No   
1           1  KX64629  California              2228.525238       No   
2           2  LZ68649  Washington             14947.917300       No   
3           3  XL78013      Oregon             22332.439460      Yes   
4           4  QA50777      Oregon              9025.067525       No   

   coverage education effective_to_date employmentstatus gender  ...  \
0     Basi

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [6]:
pivot_revenue = df.pivot_table(
    index="sales_channel",       # las filas serán los canales (Branch, Web, etc.)
    values="total_claim_amount", # el valor que queremos sumar
    aggfunc="sum"                # qué operación hacer → suma total
).round(2)                       # redondear a 2 decimales

print("=== REVENUE POR CANAL DE VENTAS ===")
print(pivot_revenue)

=== REVENUE POR CANAL DE VENTAS ===
               total_claim_amount
sales_channel                    
Agent                  1810226.82
Branch                 1301204.00
Call Center             926600.82
Web                     706600.04


In [7]:
canal_top = pivot_revenue["total_claim_amount"].idxmax()
print(f"\nEl canal con más revenue es: {canal_top}")


El canal con más revenue es: Agent


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [8]:
# Filas son el nivel de educación
# y las columnas son el género (M y F)
pivot_clv = df.pivot_table(
    index="education",                 # filas = nivel educativo
    columns="gender",                  # columnas = género
    values="customer_lifetime_value",  # valor a calcular
    aggfunc="mean"                     # operación → promedio
).round(2)

print("=== CLV PROMEDIO POR EDUCACIÓN Y GÉNERO ===")
print(pivot_clv)

=== CLV PROMEDIO POR EDUCACIÓN Y GÉNERO ===
gender                      F        M
education                             
Bachelor              7874.27  7703.60
College               7748.82  8052.46
Doctor                7328.51  7415.33
High School or Below  8675.22  8149.69
Master                8157.05  8168.83


In [9]:
print("\nNivel educativo con mayor CLV promedio:")
print(pivot_clv.mean(axis=1).idxmax())


Nivel educativo con mayor CLV promedio:
High School or Below


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [10]:
# Convertir la columna de fecha a tipo fecha
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])

# Extraer el nombre del mes
df["month"] = df["effective_to_date"].dt.month_name()

In [11]:
pivot_complaints = df.pivot_table(
    index="policy_type",               # filas = tipo de póliza
    columns="month",                   # columnas = mes
    values="number_of_open_complaints",# valor a contar
    aggfunc="sum"                      # suma de quejas
).round(0)

print("=== QUEJAS POR PÓLIZA Y MES (formato ancho) ===")
print(pivot_complaints)


=== QUEJAS POR PÓLIZA Y MES (formato ancho) ===
month           February  January
policy_type                      
Corporate Auto     385.0    443.0
Personal Auto     1454.0   1728.0
Special Auto        95.0     87.0


In [12]:
# reset_index() convierte el índice en columna normal
# melt() "desapila" las columnas de mes en filas
pivot_long = pivot_complaints.reset_index().melt(
    id_vars="policy_type",     # esta columna se queda fija
    var_name="month",          # nombre de la nueva columna de meses
    value_name="complaints"    # nombre de la nueva columna de valores
)

print("\n=== QUEJAS POR PÓLIZA Y MES (formato largo) ===")
print(pivot_long.sort_values("complaints", ascending=False).head(10))


=== QUEJAS POR PÓLIZA Y MES (formato largo) ===
      policy_type     month  complaints
4   Personal Auto   January      1728.0
1   Personal Auto  February      1454.0
3  Corporate Auto   January       443.0
0  Corporate Auto  February       385.0
2    Special Auto  February        95.0
5    Special Auto   January        87.0
